In [3]:
import sys
import os

sys.path.insert(1,'/data1/gridsan/groups/manthiram_lab/Utils')
import utils
from utils import make_xyz_from_output,checkvibs, check, no_to_symbol

import numpy as np
import pandas as pd
import pickle
import re
import importlib
import cclib
from os.path import exists
import re
import subprocess
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.pyplot as plt
import plot as pl
import pathlib
from datetime import datetime, timedelta
import pickle as pkl

from rdkit import Chem

In [4]:
importlib.reload(pl)
importlib.reload(utils)

<module 'utils' from '/data1/gridsan/groups/manthiram_lab/vHTP/Code/Utils/utils.py'>

In [5]:
#Define Current directory
cwd= '/home/gridsan/jmaalouf/vHTP/Code/ylide/Library'

# Energy Functions

In [6]:
def calcE(G_ox,G_red,n=1):
    """ 
    Calculations the redox potential of a reaction using dG=-nFE-4.28 to calculate vs SHE.
    4.28 is SHE vs vacuum.

    Arguments:
    G_ox (int): Gibbs free energy of oxidized species in hartree references to vacuum.
    G_red (int): Gibbs free energy of reduced species in hartree references to vacuum.

    Returns:
    E vs SHE (int): Redox potential in volts references vs SHE. 

    """  
    F=96485 #Faraday's constant [=] C/mol
    dG=G_ox-G_red
    E= dG*2625.5*1000/(n*F) - 4.28 #Volts
    return E




def calcDPFE(G_ylide_h,G_ylide,solvent='gas'):
    """ 
    Calculations the deprotonation Free Energy of a reaction, specifically going from ylide_h to ylide

    Arguments:
    ylide_h (int): Gibbs free energy of protonated ylide in hartree references to vacuum.
    ylide (int): Gibbs free energy of ylide in hartree references to vacuum.

    Returns:
    DPFE (int): Deprotonation Free Energy in kJ/mol. 

    """ 
    
    #Dictionaory of H+ energies at the M062X 
    proton_dict={
    'gas': -0.010000,
    'acetonitrile': -0.223572
    }
    
    possible_solvents=['gas','acetonitrile']
    assert solvent in possible_solvents, "VALID SOLVENT NOT SPECIFIED"
    
    return -(G_ylide_h-G_ylide-proton_dict[solvent])*2625.5    


def calcGHBind(G_ylide_h,G_ylide_rad,solvent='gas'):
    """ 
    Calculations the Hydrogen atom Binding Energy of a reaction, specifically going from ylide_rad to ylide_h

    Arguments:
    ylide_h (int): Gibbs free energy of protonated ylide in hartree references to vacuum.
    ylide_rad (int): Gibbs free energy of ylide radical in hartree references to vacuum.

    Returns:
    GHBind (int): Hydrogen Atom Binding energy of an ylide in kJ/mol. 

    """ 
    
    #Dictionaory of H atom energies at the M062X 
    Hatom_dict={
    'gas': -0.508484,
    'acetonitrile': -0.508404
    }
    
    H2_dict={
    'gas': -1.169741,
    'acetonitrile':-1.169345
    }
    
    possible_solvents=['gas','acetonitrile']
    
    assert solvent in possible_solvents, "VALID SOLVENT NOT SPECIFIED"
    
    return (G_ylide_h-G_ylide_rad-H2_dict[solvent]/2)*2625.5

In [7]:
df=pd.read_csv('sheets/Ylides_YlideRads_Cleaned.csv')
y=df['Ylides'].to_list()
yr=df['Ylides Rad'].to_list()
yh=df['Ylides H'].to_list()

## Get Num Atoms

In [8]:
num_atoms=[]
num_atoms_total=[]
for i, s in enumerate(tqdm(y)):
    mol=Chem.MolFromSmiles(s)
    num_atoms.append(mol.GetNumAtoms())
    mol=Chem.AddHs(mol)
    num_atoms_total.append(mol.GetNumAtoms())

100%|██████████| 19097/19097 [00:01<00:00, 10134.13it/s]


In [6]:
with open(f"pickled_data/num_atoms.pkl", "wb") as file: 
    pickle.dump(num_atoms, file)
    
with open(f"pickled_data/num_atoms_total.pkl", "wb") as file: 
    pickle.dump(num_atoms_total, file)

# Check for Failed Calcs

In [8]:
trueChoices = ['true', 't','T']
falseChoices = ['false', 'f','F']

inp=input("Do you want to resubmit the memory allocation errors? (T/F)").lower()

assert inp in trueChoices or inp in falseChoices
if inp in trueChoices:
    resubmit=True
elif inp in falseChoices:
    resubmit=False

orig_dir='/home/gridsan/groups/manthiram_lab/vHTP/Code/ylide/Library'
ylide_type=[['ylide',''],['ylide_rad','_y_h_opt'],['ylide_h','']]
#ylide_type=[['ylide_rad','_y_h_opt']]
#ylide_type=[['ylide','_gas_preopt'],['ylide_rad','_gas_preopt'],['ylide_h','_gas_preopt']]
#ylide_type=[['ylide_rad','_gas_preopt']]
#ylide_type=[['ylide_rad','_y_h_opt']]
#ylide_type=[['ylide','_gas_preopt'],['ylide_h','_gas_preopt']]
failed_dict={}

functional='M062X'
basis= 'Def2TZVP'
solvorgas='gas'
solvmethod='SMD'
solvent='acetonitrile'

start=1000
stop=10000
n=stop-start
error_occurred=False

for v,y_list in enumerate(ylide_type):
    
    y=y_list[0]
    sub_suff=y_list[1]
    
    failed_dict[f'{y}{sub_suff}']=[]
    
    for i in tqdm(range(start,stop)):
        n_str=str(i).zfill(7)
        
        
        assert solvorgas == 'gas' or solvorgas=='solv','SOLVORGAS NOT SET TO A VALID VALUE'
        
        if solvorgas=='gas':
            path=pathlib.Path(os.path.join(orig_dir, 'Calcs', str(i).zfill(7) , y, functional, basis,'gas', f'{str(i).zfill(7)}_{y}{sub_suff}.log'))
            
        elif solvorgas=='solv':
            path=pathlib.Path(os.path.join(orig_dir, 'Calcs',str(i).zfill(7) , y, functional, basis,solvent,solvmethod, f'{str(i).zfill(7)}_{y}{sub_suff}.log'))
        
        par_path=path.parent.absolute()
        
        if exists(path):
            
            if check(path, 'Normal termination of Gaussian', count=2)==False:
                error_occurred=True
                
                if check(str(path),'galloc:  could not allocate memory.'):
                    
                    print(f'MEMRORY ALLOCATION ERROR WITH {y}{sub_suff} MOLECULE {i}')
                    failed_dict[f'{y}{sub_suff}'].append(i)
                    
                    if resubmit:
                        os.chdir(par_path)
                        subprocess.run(f'sbatch {n_str}_{y}{sub_suff}.sh',shell=True)
                        os.chdir(orig_dir)
                                
                elif check(str(path),'Converged',120):
                    print(f'GEOMETRY OPT ERROR WITH {y}{sub_suff} MOLECULE {i}')
                    failed_dict[f'{y}{sub_suff}'].append(i)
                    
                else:
                    print(f'UNKNOWN ERROR, {y}{sub_suff} MOLECULE {i} CALCULATION DID NOT FINSIH')
                    failed_dict[f'{y}{sub_suff}'].append(i)
                    
                    if resubmit:
                        os.chdir(par_path)
                        #subprocess.run(f'sbatch {n_str}_{y}{sub_suff}.sh',shell=True)
                        os.chdir(orig_dir)
        else:
            print(f'{y} MOLECULE {i}{sub_suff} HAS NOT BEEN SUBMITTED')
            failed_dict[f'{y}{sub_suff}'].append(i)
            
            if resubmit:
                os.chdir(par_path)
                subprocess.run(f'sbatch {n_str}_{y}{sub_suff}.sh',shell=True)
                os.chdir(orig_dir)
                
    if error_occurred==False:
        print(f'NO ERRORS WITH {y}{sub_suff}')
        
        
    error_occurred=False
    print('')
    


Do you want to resubmit the memory allocation errors? (T/F) f


  1%|          | 85/9000 [00:09<10:48, 13.74it/s]

GEOMETRY OPT ERROR WITH ylide MOLECULE 1083


  4%|▍         | 362/9000 [00:26<06:31, 22.06it/s]

UNKNOWN ERROR, ylide MOLECULE 1356 CALCULATION DID NOT FINSIH
MEMRORY ALLOCATION ERROR WITH ylide MOLECULE 1357
MEMRORY ALLOCATION ERROR WITH ylide MOLECULE 1358
MEMRORY ALLOCATION ERROR WITH ylide MOLECULE 1359
MEMRORY ALLOCATION ERROR WITH ylide MOLECULE 1360
MEMRORY ALLOCATION ERROR WITH ylide MOLECULE 1361
MEMRORY ALLOCATION ERROR WITH ylide MOLECULE 1362
MEMRORY ALLOCATION ERROR WITH ylide MOLECULE 1363


...

ylide_h MOLECULE 9889 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9890 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9891 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9892 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9893 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9894 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9895 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9896 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9897 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9898 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9899 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9900 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9901 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9902 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9903 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9904 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9905 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9906 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9907 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9908 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9909 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9910 HAS NOT BEEN SUBMITTED
ylide_h MO

 99%|█████████▉| 8949/9000 [05:08<00:00, 138.41it/s]

ylide_h MOLECULE 9948 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9949 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9950 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9951 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9952 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9953 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9954 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9955 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9956 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9957 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9958 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9959 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9960 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9961 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9962 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9963 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9964 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9965 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9966 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9967 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9968 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9969 HAS NOT BEEN SUBMITTED
ylide_h MO

100%|██████████| 9000/9000 [05:08<00:00, 29.16it/s] 

ylide_h MOLECULE 9980 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9981 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9982 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9983 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9984 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9985 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9986 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9987 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9988 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9989 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9990 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9991 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9992 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9993 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9994 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9995 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9996 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9997 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9998 HAS NOT BEEN SUBMITTED
ylide_h MOLECULE 9999 HAS NOT BEEN SUBMITTED



In [10]:
len(failed_dict['ylide'])

5864

In [9]:
#os.chdir(cwd)
with open(f"/data1/groups/manthiram_lab/vHTP/Code/ylide/Library/python_scripts/failed_dict_{solvorgas}.pkl", "wb") as file: 
    pickle.dump(failed_dict, file)

In [17]:
os.chdir(cwd)
with open(f"pickled_data/failed_dict_{solvorgas}.pkl", "wb") as file: 
    pickle.dump(failed_dict, file)
#failed_dict

In [11]:
for key, value in failed_dict.items():
    print(f'{key} has {len(value)} failures and a fail percentage of {len(value)/n*100}%')

ylide has 5864 failures and a fail percentage of 65.15555555555555%
ylide_rad_y_h_opt has 9000 failures and a fail percentage of 100.0%
ylide_h has 5894 failures and a fail percentage of 65.48888888888888%


# Extract Data

In [19]:
functional='M062X'
basis= 'Def2TZVP'
solvorgas='gas'
solvmethod='SMD'

orig_dir='/home/gridsan/groups/manthiram_lab/vHTP/Code/ylide/Library'
if solvorgas=='gas':
    solvent=='gas'
    ylide_type=[['ylide',''],['ylide_rad','_y_h_opt'],['ylide_h','']]
    #ylide_type=[['ylide_h','']]
else:
    solvent='acetonitrile'
    ylide_type=[['ylide','_gas_preopt'],['ylide_rad','_gas_preopt'],['ylide_h','_gas_preopt'] ]

start=0
stop=1000
n=stop-start
data=np.empty((n,10))
atom_coords=[] #Dim1= Ylide Dim2=Ylide Rad Dim3= Ylide_H
atom_labels=[] #Dim1= Ylide Dim2=Ylide Rad Dim3= Ylide_H
wall_time=[] #Calculation time in seconds

for v,y_list in enumerate(ylide_type):
    
    y=y_list[0]
    sub_suff=y_list[1]
    
    temp_atom_coords=[]
    temp_atom_labels=[]
    temp_wall_time=[]
    
    for k,i in enumerate(tqdm(range(start,stop))):
        n_str=str(i).zfill(7)

        
        
        assert solvorgas == 'gas' or solvorgas=='solv','SOLVORGAS NOT SET TO A VALID VALUE'
        
        if solvorgas=='gas':
            path=pathlib.Path(os.path.join(orig_dir, 'Calcs', str(i).zfill(7) , y, functional, basis,'gas', f'{str(i).zfill(7)}_{y}{sub_suff}.log'))
            
        elif solvorgas=='solv':
            path=pathlib.Path(os.path.join(orig_dir, 'Calcs',str(i).zfill(7) , y, functional, basis,solvent,solvmethod, f'{str(i).zfill(7)}_{y}{sub_suff}.log'))
        
        
        
        if exists(path): #If calculation was run
            
            if check(path, 'Normal termination of Gaussian',count=2): #If calculation was run successfully
                try:
                    
                    temp_data=cclib.io.ccread(str(path))

                    meta_data=temp_data.metadata
                    data[k,2*v]=temp_data.enthalpy
                    data[k,2*v+1]=temp_data.freeenergy
                    data[k,6]=checkvibs(temp_data.vibfreqs,asint=True)
                    temp_atom_coords.append(temp_data.atomcoords[-1,:,:])
                    temp_atom_labels.append([no_to_symbol(x) for x in temp_data.atomnos])
                    temp_wall_time.append(sum(meta_data['wall_time'][i].seconds for i in range(len(meta_data['wall_time']))))
                    
                    continue
                except Exception as e:
                    exc_type, exc_obj, exc_tb = sys.exc_info()
                    fname = os.path.split(exc_tb.tb_frame.f_code.co_filename)[1]
                    print(e)       
                    print(exc_type, fname, exc_tb.tb_lineno)

                    data[k,2*v]= 1
                    data[k,2*v+1]= 1
                    data[k,6]=-1
                    temp_atom_coords.append([])
                    temp_atom_labels.append([])
                    temp_wall_time.append(0)
                    continue
            else: #Run unseccessfully
                data[k,2*v]= 1
                data[k,2*v+1]= 1
                data[k,6]=-1
                temp_atom_coords.append([])
                temp_atom_labels.append([])
                temp_wall_time.append(0)
                continue
        else:
            data[k,2*v]= 1
            data[k,2*v+1]= 1
            data[k,6]=-1
            temp_atom_coords.append([])
            temp_atom_labels.append([])
            temp_wall_time.append(0)
            continue

    atom_coords.append(temp_atom_coords)
    atom_labels.append(temp_atom_labels)
    wall_time.append(temp_wall_time)

 32%|███▏      | 323/1000 [01:19<02:39,  4.25it/s]

'ccData_optdone_bool' object has no attribute 'vibfreqs'
<class 'AttributeError'> <ipython-input-19-bbed533df0ed> 57


 42%|████▏     | 419/1000 [01:41<02:39,  3.63it/s]

'ccData_optdone_bool' object has no attribute 'vibfreqs'
<class 'AttributeError'> <ipython-input-19-bbed533df0ed> 57


 48%|████▊     | 478/1000 [01:55<02:19,  3.75it/s]

...

 89%|████████▊ | 887/1000 [04:16<00:41,  2.71it/s]

'ccData_optdone_bool' object has no attribute 'vibfreqs'
<class 'AttributeError'> <ipython-input-19-bbed533df0ed> 57


 97%|█████████▋| 970/1000 [04:42<00:06,  4.56it/s]

'ccData_optdone_bool' object has no attribute 'vibfreqs'
<class 'AttributeError'> <ipython-input-19-bbed533df0ed> 57
'ccData_optdone_bool' object has no attribute 'vibfreqs'
<class 'AttributeError'> <ipython-input-19-bbed533df0ed> 57


100%|██████████| 1000/1000 [03:55<00:00,  4.25it/s]


In [103]:
len(atom_coords[2])

500

In [19]:
np.mean(wall_time[2][:])

1876.582

#### Wall Times For Different Triples Conditons

[5,1,1] wall_time: [1816, 2098, 2524, 2864, 1405, 1256, 1628, 1559, 1741, 1783] <br>
[10,1,2] wall_time: [[1800, 0, 0, 0, 1398, 1248, 1618, 1560, 1732, 1787]] <br>
[5,2,1] wall_time: [3784, 0, 5337, 0, 0, 2829, 0, 3480, 3763, 0] <br>

There seems to be no effect from number of threads, it seems that we still have an effect from RAM and it scales pretty linearly so I will use 1 node per calculation

In [20]:
print('CALCULATING REDOX POTENTIALS')    
for i, v in enumerate(tqdm(range(np.shape(data)[0]))):
    data[i,7]=calcE(data[i,3],data[i,1])
    
    
print('CALCULATING DPFE')
for i, v in enumerate(tqdm(range(np.shape(data)[0]))):
    data[i,8]=calcDPFE(data[i,5],data[i,1],solvent=solvent)

print('CALCULATING GHBind ')
for i, v in enumerate(tqdm(range(np.shape(data)[0]))):
    data[i,9]=calcGHBind(data[i,5],data[i,3],solvent='gas')
    

100%|██████████| 1000/1000 [00:00<00:00, 703270.29it/s]

CALCULATING REDOX POTENTIALS
CALCULATING DPFE
CALCULATING GHBind 


In [12]:
!pwd

/home/gridsan/groups/manthiram_lab/vHTP/Code/ylide/Library


In [21]:
#Save Extracted data
cwd='/home/gridsan/jmaalouf/vHTP/Code/ylide/Library'
os.chdir(cwd)



with open(f"pickled_data/wall_time_{solvorgas}.pkl", "wb") as file: 
    pickle.dump(wall_time, file)

with open(f"pickled_data/atom_coords_{solvorgas}.pkl", "wb") as file: 
    pickle.dump(atom_coords, file)

with open(f"pickled_data/atom_labels.pkl","wb") as file:

    pickle.dump(atom_labels, file)

np.save(f'pickled_data/data_{solvorgas}.npy', data)


In [58]:
df=pd.DataFrame(data, columns=['ylide_enthalpy','ylide_freeenergy','ylide_rad_enthalpy','ylide_rad_freeenergy','ylide_h_enthalpy','ylide_h_freeenergy','Vibs All Positivse','Redox Potential (V vs SHE)','Deprotonation Free Energy (kJ/mol)'])

In [43]:
#df.to_csv('ylide_calcs_6-311++G**.csv')